In [2]:
from pathlib import Path

import pandas as pd

In [3]:
Path.cwd()

WindowsPath('d:/学习/大学/大二上/zyy大创/导师科研项目/notebooks')

In [5]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("项目根目录：", PROJECT_ROOT)
print("原始数据目录：", DATA_RAW)

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

项目根目录： d:\学习\大学\大二上\zyy大创\导师科研项目
原始数据目录： d:\学习\大学\大二上\zyy大创\导师科研项目\data\raw


In [6]:
raw_files = sorted(
    p.name
    for p in DATA_RAW.iterdir()
    if p.is_file()
)

raw_files

['firm_financials.xlsx', 'firm_profile.csv', 'patents.csv']

In [7]:
financials = pd.read_excel(DATA_RAW / "firm_financials.xlsx")

In [8]:
financials.shape

(260, 11)

In [9]:
financials.columns.tolist()

['stock_code',
 'company_name',
 'year',
 'total_assets',
 'total_liabilities',
 'revenue',
 'net_profit',
 'cash',
 'rd_expense',
 'roe',
 'employees']

In [12]:
financials.dtypes

stock_code           float64
company_name             str
year                  object
total_assets          object
total_liabilities     object
revenue               object
net_profit           float64
cash                 float64
rd_expense            object
roe                   object
employees            float64
dtype: object

In [13]:
financials.info()

<class 'pandas.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   stock_code         259 non-null    float64
 1   company_name       260 non-null    str    
 2   year               260 non-null    object 
 3   total_assets       260 non-null    object 
 4   total_liabilities  260 non-null    object 
 5   revenue            260 non-null    object 
 6   net_profit         259 non-null    float64
 7   cash               259 non-null    float64
 8   rd_expense         260 non-null    object 
 9   roe                260 non-null    object 
 10  employees          259 non-null    float64
dtypes: float64(4), object(6), str(1)
memory usage: 30.1+ KB


In [14]:
financials.isna().sum()

stock_code           1
company_name         0
year                 0
total_assets         0
total_liabilities    0
revenue              0
net_profit           1
cash                 1
rd_expense           0
roe                  0
employees            1
dtype: int64

In [15]:
text_like_columns = financials.select_dtypes(include=["object", "string"]).columns.tolist()
text_like_columns

['company_name',
 'year',
 'total_assets',
 'total_liabilities',
 'revenue',
 'rd_expense',
 'roe']

In [16]:
for column in text_like_columns:
    values = (
        financials[column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    print(f"\n{column}:")
    print(values[:15])


company_name:
<ArrowStringArray>
[   '华辰科技股份有限公司',      '华辰科技有限公司',  'ST华辰科技股份有限公司', '*ST华辰科技股份有限公司',
    '新岳科技股份有限公司',    '海川科技股份有限公司',    '中盛科技股份有限公司',    '宏远科技股份有限公司',
    '智恒科技股份有限公司',    '瑞华科技股份有限公司',    '星海科技股份有限公司',    '天成科技股份有限公司',
      '天成科技有限公司',    '博创科技股份有限公司',  'ST博创科技股份有限公司']
Length: 15, dtype: str

year:
<ArrowStringArray>
['2020', '2021', '2022', '2023年', '2024', '2025', '2023']
Length: 7, dtype: str

total_assets:
<ArrowStringArray>
[   '44813001602',    '51047456089',    '53781621074',     '1934677743',
    '48924685296',    '42036826746',    '20414025335',     '7787566354',
 '31,241,347,993',    '21721819382',     '9274052273',    '51847529952',
     '9791639194',    '11822974703',    '37487571181']
Length: 15, dtype: str

total_liabilities:
<ArrowStringArray>
['23908847993', '30247752226', '40472674936',   '476851735', '37663675836',
 '14386976040', '13997378864',  '4122647494', '18448723676',  '4181453282',
  '2248802709', '23626999177',  '2128754301',  '78555599

In [17]:
pseudo_missing_tokens = [
    "",
    "-",
    "--",
    "NA",
    "N/A",
    "NULL",
    "None",
]

for column in text_like_columns:
    values = financials[column].astype("string").str.strip()

    counts = (
        values[values.isin(pseudo_missing_tokens)]
        .value_counts()
    )

    if not counts.empty:
        print(f"\n{column}:")
        print(counts)


total_liabilities:
total_liabilities
-    1
Name: count, dtype: int64[pyarrow]

rd_expense:
rd_expense
-    1
Name: count, dtype: int64[pyarrow]


In [18]:
year_text = financials["year"].astype("string").str.strip()

invalid_year_format = year_text[
    ~year_text.str.fullmatch(r"\d{4}", na=False)
]

invalid_year_format.value_counts()

year
2023年    6
Name: count, dtype: int64[pyarrow]

In [19]:
numeric_like_columns = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "rd_expense",
    "roe",
]

for column in numeric_like_columns:
    values = financials[column].astype("string").str.strip()

    comma_count = values.str.contains(",", regex=False, na=False).sum()
    percent_count = values.str.contains("%", regex=False, na=False).sum()

    print(
        f"{column}: "
        f"千位逗号={comma_count}, "
        f"百分号={percent_count}"
    )

total_assets: 千位逗号=1, 百分号=0
total_liabilities: 千位逗号=0, 百分号=0
revenue: 千位逗号=1, 百分号=0
rd_expense: 千位逗号=0, 百分号=0
roe: 千位逗号=0, 百分号=2


In [20]:
for column in numeric_like_columns:
    values = financials[column].astype("string").str.strip()

    # 仅用于审计的临时副本，不修改 financials
    audit_values = values.mask(values.isin(pseudo_missing_tokens))

    # 暂时去掉已知格式符号，测试剩余内容能否解释为数字
    audit_values = (
        audit_values
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
    )

    parsed = pd.to_numeric(audit_values, errors="coerce")

    unknown_mask = audit_values.notna() & parsed.isna()
    unknown_values = values[unknown_mask].value_counts()

    print(f"\n{column}: 无法解释为数字的值 = {unknown_mask.sum()}")

    if not unknown_values.empty:
        print(unknown_values)


total_assets: 无法解释为数字的值 = 0

total_liabilities: 无法解释为数字的值 = 0

revenue: 无法解释为数字的值 = 0

rd_expense: 无法解释为数字的值 = 0

roe: 无法解释为数字的值 = 0


In [21]:
exact_duplicate_count = financials.duplicated().sum()

exact_duplicate_count

np.int64(10)

In [22]:
key_columns = ["stock_code", "year"]

duplicate_key_count = financials.duplicated(
    subset=key_columns,
    keep=False
).sum()

duplicate_key_count

np.int64(40)

In [23]:
duplicate_key_rows = (
    financials.loc[
        financials.duplicated(
            subset=key_columns,
            keep=False
        )
    ]
    .sort_values(key_columns)
)

duplicate_key_rows[
    [
        "stock_code",
        "company_name",
        "year",
        "total_assets",
        "revenue",
        "net_profit",
    ]
]

,stock_code,company_name,year,total_assets,revenue,net_profit
0,1.0,华辰科技股份有限公司,2020,44813001602,8788852319,8.233325e+08
240,1.0,华辰科技股份有限公司,2020,44813001602,8788852319,8.233325e+08
10,2.0,新岳科技股份有限公司,2024,9274052273,44523089214,1.471324e+10
250,2.0,新岳科技股份有限公司,2024,9274052273,44523089214,1.500893e+09
17,3.0,海川科技股份有限公司,2025,51903132738,6197206185,1.456729e+08
241,3.0,海川科技股份有限公司,2025,51903132738,6197206185,1.456729e+08
27,5.0,宏远科技股份有限公司,2023年,33191213417,25711723804,2.855787e+09
251,5.0,宏远科技股份有限公司,2023年,33191213417,25711723804,1.766409e+09
34,6.0,智恒科技股份有限公司,2024,24192980006,24585101604,4.884060e+09
242,6.0,智恒科技股份有限公司,2024,24192980006,24585101604,4.884060e+09


In [24]:
financials.loc[
    financials["stock_code"].isna(),
    ["stock_code", "company_name", "year"]
]

,stock_code,company_name,year
21,NaN,中盛科技股份有限公司,2023年


## 阶段性数据审计结论

本 Notebook 对 `firm_financials.xlsx` 原始财务面板训练数据进行了初步质量审计，当前未对原始数据执行清洗或删除操作。

### 数据基本结构

- 原始数据规模：260 行，11 个变量
- 理论观测单位：公司 × 年份（firm-year）
- 潜在主键：`stock_code + year`

### 已识别的数据质量问题

1. **原生缺失值**
   - `stock_code`：1 条
   - `net_profit`：1 条
   - `cash`：1 条
   - `employees`：1 条

2. **伪缺失值**
   - `total_liabilities` 中 `"-"`：1 条
   - `rd_expense` 中 `"-"`：1 条

3. **格式不一致**
   - `year` 中 `"2023年"`：6 条
   - `total_assets` 中千位逗号格式：1 条
   - `revenue` 中千位逗号格式：1 条
   - `roe` 中百分号格式：2 条
   - 对已知格式符号进行审计性标准化后，未发现其他无法解释为数值的文本

4. **主键及重复问题**
   - 完全重复的后续记录：10 条
   - 位于重复 `stock_code + year` 主键组中的记录：40 条
   - 说明除完全重复记录外，还存在同一 firm-year 财务值不一致的冲突记录
   - `stock_code` 缺失：1 条，因此该记录当前无法形成完整 firm-year 主键

### 当前结论

原始数据中的主要质量问题已经能够被系统识别并分类，包括真实缺失、伪缺失、格式异常、完全重复以及主键冲突。

下一阶段将在保留原始数据不变的前提下，建立明确的数据清洗规则，并输出标准化后的 processed 数据集。

In [27]:
audit_summary = pd.DataFrame(
    [
        ["数据规模", "rows", 260],
        ["数据规模", "columns", 11],
        ["真实缺失", "stock_code", 1],
        ["真实缺失", "net_profit", 1],
        ["真实缺失", "cash", 1],
        ["真实缺失", "employees", 1],
        ["伪缺失", "total_liabilities: '-'", 1],
        ["伪缺失", "rd_expense: '-'", 1],
        ["格式异常", "year: '2023年'", 6],
        ["格式异常", "total_assets: 千位逗号", 1],
        ["格式异常", "revenue: 千位逗号", 1],
        ["格式异常", "roe: 百分号", 2],
        ["重复问题", "完全重复后续记录", int(exact_duplicate_count)],
        ["主键问题", "重复 firm-year 中的记录", int(duplicate_key_count)],
        ["主键问题", "stock_code 缺失", 1],
    ],
    columns=["category", "issue", "count"],
)

audit_summary

,category,issue,count
0,数据规模,rows,260
1,数据规模,columns,11
2,真实缺失,stock_code,1
3,真实缺失,net_profit,1
4,真实缺失,cash,1
5,真实缺失,employees,1
6,伪缺失,total_liabilities: '-',1
7,伪缺失,rd_expense: '-',1
8,格式异常,year: '2023年',6
9,格式异常,total_assets: 千位逗号,1
